# Smart Batching Search - Testing Notebook

This notebook demonstrates and tests the smart batching search functionality.

## Features
- **Planning**: Organize search using smart batching
- **Execution**: Execute search with proportional sampling
- **Rate Limiting**: Configurable requests per minute
- **Parallel Processing**: Efficient parallel execution

## Configuration

**Environment Variables:** This notebook loads configuration from a `.env` file in the `Smart_Batching` directory.

Create a `.env` file with:
```
BIGDATA_API_KEY=your_api_key_here
BIGDATA_API_BASE_URL=https://api.bigdata.com
```

**Options for API_BASE_URL:**
- `https://api.bigdata.com` (production - default)

**Note:** You must restart the kernel and run cells from the beginning if you change the API Base URL, as it's read at import time.

## 1. Load Environment Variables and Setup

**IMPORTANT:** Load `.env` file and set API_BASE_URL here before importing modules, as it's read at import time.

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
# Look for .env in the current directory (Smart_Batching folder)
env_path = Path.cwd() / ".env"
if env_path.exists():
    load_dotenv(env_path)
    print(f"✅ Loaded environment variables from {env_path}")
else:
    print(f"⚠️  .env file not found at {env_path}")
    print("   Using environment variables or defaults")

# Set API base URL BEFORE importing search_function
# This is important because smart_batching_config reads it at import time
API_BASE_URL = os.getenv("BIGDATA_API_BASE_URL", "https://api.bigdata.com")
os.environ["BIGDATA_API_BASE_URL"] = API_BASE_URL

print(f"✅ API Base URL configured: {API_BASE_URL}")

# Add current directory to path
sys.path.insert(0, str(Path.cwd()))

from search_function import (
    plan_search,
    execute_search,
    deduplicate_documents,
    save_plan,
    load_plan,
    load_universe_from_csv
)
from output_converter import convert_to_dataframe

print("✅ Imports successful")

✅ Loaded environment variables from /home/fricigliano/git/bigdata/github/bigdata-cookbook/Smart_batching/.env
✅ API Base URL configured: https://api.bigdata.com
✅ Imports successful


## 2. Configuration

In [2]:
# Configuration
# Note: API_BASE_URL and API_KEY are loaded from .env file in cell 2
# To change them, edit the .env file or set environment variables:
#   export BIGDATA_API_KEY='your_api_key_here'
#   export BIGDATA_API_BASE_URL='https://api.bigdata.com'

# API Key (loaded from .env in cell 2)
API_KEY = os.getenv("BIGDATA_API_KEY")

if not API_KEY:
    print("⚠️  BIGDATA_API_KEY not set. Please set it in your .env file:")
    print("   BIGDATA_API_KEY=your_api_key_here")
    print("   Or set environment variable: export BIGDATA_API_KEY='your_api_key_here'")
else:
    print(f"✅ API Key configured: {API_KEY[:8]}...{API_KEY[-4:]}")

# Test parameters
TEST_TEXT = "Decline in customer confidence in the company"
TEST_UNIVERSE_CSV = "test_data/us_top3000.csv"  # full universe
# TEST_UNIVERSE_CSV = "sample_universe.csv"  # small test universe
TEST_START_DATE = "2021-01-01"
TEST_END_DATE = "2021-06-30"
TEST_CHUNK_PERCENTAGE = 0.1  # 10% of total chunks

print(f"\n📝 Test Configuration:")
print(f"   API Base URL: {API_BASE_URL}")
print(f"   Text: '{TEST_TEXT}'")
print(f"   Universe: {TEST_UNIVERSE_CSV}")
print(f"   Date Range: {TEST_START_DATE} to {TEST_END_DATE}")
print(f"   Chunk Percentage: {TEST_CHUNK_PERCENTAGE*100:.0f}%")

✅ API Key configured: bd_v1_NW...2252

📝 Test Configuration:
   API Base URL: https://api.bigdata.com
   Text: 'Decline in customer confidence in the company'
   Universe: test_data/us_top3000.csv
   Date Range: 2021-01-01 to 2021-06-30
   Chunk Percentage: 10%


## 3. Test Universe Loading

In [3]:
# Test loading universe from CSV
try:
    companies = load_universe_from_csv(TEST_UNIVERSE_CSV)
    print(f"✅ Loaded {len(companies)} companies from {TEST_UNIVERSE_CSV}")
    print(f"   First 5 companies: {companies[:5]}")
except Exception as e:
    print(f"❌ Error loading universe: {e}")

2026-01-27 13:24:55,043 - INFO - Loaded 4731 entity IDs from test_data/us_top3000.csv
✅ Loaded 4731 companies from test_data/us_top3000.csv
   First 5 companies: ['00067A', '001F1B', '002A99', '00326D', '003B70']


## 4. Step 1: Plan Search

In [ ]:
# Plan the search
if API_KEY:
    print("📋 Planning search...")
    print("-" * 80)
    
    try:
        companies = load_universe_from_csv(TEST_UNIVERSE_CSV)

        plan = plan_search(
            text=TEST_TEXT,
            companies=companies,
            start_date=TEST_START_DATE,
            end_date=TEST_END_DATE,
            api_key=API_KEY,
            api_base_url=API_BASE_URL
        )
        
        print(f"\n✅ Planning complete!")
        print(f"   Total expected chunks: {plan['total_expected_chunks']:,}")
        print(f"   Number of baskets: {len(plan['baskets'])}")
        
        if plan.get('planning_metadata'):
            metadata = plan['planning_metadata']
            print(f"   Total companies: {metadata.get('total_companies', 'N/A')}")
            print(f"   Companies with chunks: {metadata.get('companies_with_chunks', 'N/A')}")
            print(f"   Uses smart batching: {metadata.get('uses_smart_batching', False)}")
        
        # Show example basket
        if plan['baskets']:
            example_basket = plan['baskets'][0]
            print(f"\n   Example Basket:")
            print(f"     Basket ID: {example_basket['basket_id']}")
            print(f"     Expected chunks: {example_basket['expected_chunks']}")
            print(f"     Companies: {len(example_basket['companies'])} companies")
            print(f"     Query text: '{example_basket['query']['text']}'")
            print(f"     Max chunks in query: {example_basket['query']['max_chunks']}")
        
    except Exception as e:
        print(f"❌ Error during planning: {e}")
        import traceback
        traceback.print_exc()
        plan = None
else:
    print("⚠️  Skipping planning - API key not set")
    plan = None

📋 Planning search...
--------------------------------------------------------------------------------
2026-01-27 13:24:55,048 - INFO - Planning search for text: 'Decline in customer confidence in the company'
2026-01-27 13:24:55,049 - INFO - Date range: 2021-01-01 to 2021-06-30
2026-01-27 13:24:55,049 - INFO - Loaded 4731 companies
2026-01-27 13:24:55,049 - INFO - Using SmartBatchingPlanner for optimized batching
    Querying 4731 companies in batches of 500 (estimated 10 queries)...
      Query 1/10: Found 325 companies from universe batch (out of 500 input)
      Query 2/10: Found 345 companies from universe batch (out of 500 input)
      Query 3/10: Found 334 companies from universe batch (out of 500 input)
      Query 4/10: Found 347 companies from universe batch (out of 500 input)
      Query 5/10: Found 339 companies from universe batch (out of 500 input)
      Query 6/10: Found 360 companies from universe batch (out of 500 input)
      Query 7/10: Found 348 companies from univer

## 5. Save Plan (Optional)

In [5]:
# Save plan for later use
if plan:
    plan_file = "test_search_plan.json"
    try:
        save_plan(plan, plan_file)
        print(f"✅ Plan saved to {plan_file}")
        print(f"   You can load it later with: plan = load_plan('{plan_file}')")
    except Exception as e:
        print(f"❌ Error saving plan: {e}")

2026-01-27 13:26:08,285 - INFO - Plan saved to test_search_plan.json
✅ Plan saved to test_search_plan.json
   You can load it later with: plan = load_plan('test_search_plan.json')


## 6. Step 2: Execute Search with Proportional Sampling

In [6]:
# Execute search with proportional sampling
if plan and API_KEY:
    print("🔍 Executing search...")
    print("-" * 80)
    
    try:
        results_raw = execute_search(
            search_plan=plan,
            chunk_percentage=0.1,
            requests_per_minute=100,  # Rate limit
            api_key=API_KEY,
            api_base_url=API_BASE_URL,
        )
        
        results = deduplicate_documents(results_raw)

        print(f"\n✅ Search complete!")
        print(f"   Retrieved {len(results):,} deduplicated chunks")
            
    except Exception as e:
        print(f"❌ Error during execution: {e}")
        import traceback
        traceback.print_exc()
        results = []
else:
    print("⚠️  Skipping execution - plan or API key not available")
    results = []

🔍 Executing search...
--------------------------------------------------------------------------------
2026-01-27 13:26:08,289 - INFO - Executing search with 10.0% of chunks
2026-01-27 13:26:08,289 - INFO - Total maximum expected chunks: 21,082
2026-01-27 13:26:08,290 - INFO - Searching 218 baskets
2026-01-27 13:26:10,739 - INFO - Basket high_basket_6: Retrieved 136 documents with 165 chunks
2026-01-27 13:26:10,885 - INFO - Basket high_basket_7: Retrieved 140 documents with 163 chunks
2026-01-27 13:26:11,213 - INFO - Basket high_basket_5: Retrieved 188 documents with 205 chunks
2026-01-27 13:26:11,429 - INFO - Basket high_basket_4: Retrieved 127 documents with 201 chunks
2026-01-27 13:26:12,311 - INFO - Basket high_basket_1: Retrieved 366 documents with 410 chunks
2026-01-27 13:26:12,561 - INFO - Basket high_basket_2: Retrieved 340 documents with 375 chunks
2026-01-27 13:26:12,841 - INFO - Basket high_basket_0: Retrieved 388 documents with 442 chunks
2026-01-27 13:26:14,355 - INFO - Ba

## 7. Analyze Results

In [7]:
# Convert to DataFrame (exploded by chunk)
df = convert_to_dataframe(results)
df.head()

,date,doc_id,headline,source_id,source_name,source_rank,chunk_index,chunk_text,chunk_relevance,chunk_sentiment,entity_ids,url,reporting_entities
0,2021-03-02,028A9CA1CB876D411A9BBCCA72062616,Online Shopping Feels the Heat as Customer Sat...,5A5702,Benzinga,RANK_1,6,Walmart and Sears remain the industry bottom d...,0.620291,-0.50,"[5442E4, 5442E4, B8EF97, 713810, 713810, 76097...",,[]
1,2021-03-02,028A9CA1CB876D411A9BBCCA72062616,Online Shopping Feels the Heat as Customer Sat...,5A5702,Benzinga,RANK_1,16,"Walgreens stays below the industry average, in...",0.228817,-0.70,"[DC1A9F, CAC8D0, business,stock-prices,stock-p...",,[]
2,2021-03-02,028A9CA1CB876D411A9BBCCA72062616,Online Shopping Feels the Heat as Customer Sat...,5A5702,Benzinga,RANK_1,20,FedEx remains in the lead despite declining 3%...,0.538583,-0.30,"[6844D2, EAD6FC, EAD6FC, 760977]",,[]
3,2021-03-02,028A9CA1CB876D411A9BBCCA72062616,Online Shopping Feels the Heat as Customer Sat...,5A5702,Benzinga,RANK_1,7,Costco remains in first place for a fifth stra...,0.153063,-0.05,"[4ECD1A, 40B903, 40B903, 422CE3, 422CE3, C5C13...",,[]
4,2021-03-02,028A9CA1CB876D411A9BBCCA72062616,Online Shopping Feels the Heat as Customer Sat...,5A5702,Benzinga,RANK_1,17,After four years of near-stable customer satis...,0.390803,-0.49,"[B8EF97, 83AFD9, 08FF34, DC1A9F, DC1A9F, 49C91...",,[]


In [15]:
#Analyze results
if results:

    print("📈 Results Analysis")
    print("-" * 80)
    
    # Summary stats
    n_docs = df['doc_id'].nunique()
    n_chunks = len(df)
    print(f"\n   Total: {n_docs:,} documents, {n_chunks:,} chunks")
    
    # Relevance distribution
    if 'chunk_relevance' in df.columns and df['chunk_relevance'].notna().any():
        print(f"\n   Relevance Scores:")
        print(f"     Min: {df['chunk_relevance'].min():.3f}")
        print(f"     Max: {df['chunk_relevance'].max():.3f}")
        print(f"     Avg: {df['chunk_relevance'].mean():.3f}")
    
    # Sentiment distribution
    if 'chunk_sentiment' in df.columns and df['chunk_sentiment'].notna().any():
        sentiments = df['chunk_sentiment'].dropna()
        positive = (sentiments > 0).sum()
        negative = (sentiments < 0).sum()
        neutral = len(sentiments) - positive - negative
        print(f"\n   Sentiment Distribution:")
        print(f"     Positive: {positive} ({positive/len(sentiments)*100:.1f}%)")
        print(f"     Negative: {negative} ({negative/len(sentiments)*100:.1f}%)")
        print(f"     Neutral: {neutral} ({neutral/len(sentiments)*100:.1f}%)")
    
    # Source distribution
    if 'source_name' in df.columns:
        source_counts = df.groupby('source_name').size().sort_values(ascending=False)
        print(f"\n   Top Sources:")
        for source, count in source_counts.head(5).items():
            print(f"     {source}: {count} chunks")
    
    # Show DataFrame info
    print(f"\n   DataFrame shape: {df.shape}")
    
    # Save results
    from datetime import datetime
    results_file = f"search_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    df.to_json(results_file, orient='records', indent=2)
    print(f"\n💾 Saved to {results_file}")

else:
    df = None
    print("⚠️  No results to analyze")

📈 Results Analysis
--------------------------------------------------------------------------------

   Total: 9,903 documents, 15,369 chunks

   Relevance Scores:
     Min: 0.042
     Max: 0.620
     Avg: 0.093

   Sentiment Distribution:
     Positive: 6143 (40.0%)
     Negative: 9080 (59.1%)
     Neutral: 140 (0.9%)

   Top Sources:
     Quartr Reports: 4578 chunks
     Factset Transcripts: 2380 chunks
     Benzinga: 2331 chunks
     Quartr Transcripts: 1220 chunks
     MT Newswires: 808 chunks

   DataFrame shape: (15369, 13)

💾 Saved to search_results_20260127_140041.json


## 8. Summary

In [9]:
print("=" * 80)
print("Smart Batching Search - Test Summary")
print("=" * 80)

if plan:
    print(f"✅ Planning: SUCCESS")
    print(f"   Expected chunks: {plan['total_expected_chunks']:,}")
    print(f"   Baskets created: {len(plan['baskets'])}")
else:
    print("⚠️  Planning: Not completed")

if results:
    print(f"✅ Execution: SUCCESS")
    print(f"   Chunks retrieved: {len(results):,}")
    print(f"   Percentage used: {TEST_CHUNK_PERCENTAGE*100:.0f}%")
    if plan:
        expected = plan['total_expected_chunks']
        actual = len(results)
        if expected > 0:
            print(f"   Actual vs Expected: {actual/expected*100:.1f}%")
else:
    print("⚠️  Execution: Not completed")

print("\n" + "=" * 80)
print("Test complete!")
print("=" * 80)

Smart Batching Search - Test Summary
✅ Planning: SUCCESS
   Expected chunks: 210,828
   Baskets created: 218
✅ Execution: SUCCESS
   Chunks retrieved: 9,903
   Percentage used: 10%
   Actual vs Expected: 4.7%

Test complete!


## 9. Load Saved Plan (Optional)

In [10]:
# Load a previously saved plan
plan_file = "test_search_plan.json"

if os.path.exists(plan_file):
    try:
        loaded_plan = load_plan(plan_file)
        print(f"✅ Plan loaded from {plan_file}")
        print(f"   Total expected chunks: {loaded_plan.get('total_expected_chunks', 0):,}")
        print(f"   Number of baskets: {len(loaded_plan.get('baskets', []))}")
        print(f"\n   You can now execute with different percentages:")
        print(f"   results = execute_search(loaded_plan, chunk_percentage=0.2)")
    except Exception as e:
        print(f"❌ Error loading plan: {e}")
else:
    print(f"ℹ️  Plan file '{plan_file}' not found. Save a plan first.")

2026-01-27 13:28:26,221 - INFO - Plan loaded from test_search_plan.json
✅ Plan loaded from test_search_plan.json
   Total expected chunks: 210,828
   Number of baskets: 218

   You can now execute with different percentages:
   results = execute_search(loaded_plan, chunk_percentage=0.2)


## 10. Test Different Percentages (Optional)

In [11]:
# Test with different chunk percentages
if plan and API_KEY:
    print("📊 Testing different chunk percentages...")
    print("-" * 80)
    
    percentages = [0.05, 0.1, 0.25]
    
    for pct in percentages:
        try:
            results_raw = execute_search(
                search_plan=plan,
                chunk_percentage=pct,
                requests_per_minute=100,
                api_key=API_KEY,
                api_base_url=API_BASE_URL
            )

            results = deduplicate_documents(results_raw)

            # Count total chunks across all documents
            total_chunks = sum(len(doc.get("chunks", [])) for doc in results)
            print(f"   {pct*100:3.0f}%: {len(results):,} documents, {total_chunks:,} chunks retrieved")
        except Exception as e:
            print(f"   {pct*100:3.0f}%: Error - {e}")
else:
    print("⚠️  Skipping - plan or API key not available")

📊 Testing different chunk percentages...
--------------------------------------------------------------------------------
2026-01-27 13:28:26,225 - INFO - Executing search with 5.0% of chunks
2026-01-27 13:28:26,225 - INFO - Total maximum expected chunks: 10,541
2026-01-27 13:28:26,225 - INFO - Searching 218 baskets
2026-01-27 13:28:27,897 - INFO - Basket high_basket_2: Retrieved 182 documents with 196 chunks
2026-01-27 13:28:27,976 - INFO - Basket high_basket_7: Retrieved 72 documents with 80 chunks
2026-01-27 13:28:28,103 - INFO - Basket high_basket_6: Retrieved 72 documents with 82 chunks
2026-01-27 13:28:28,266 - INFO - Basket high_basket_3: Retrieved 102 documents with 112 chunks
2026-01-27 13:28:28,273 - INFO - Basket high_basket_5: Retrieved 92 documents with 99 chunks
2026-01-27 13:28:28,738 - INFO - Basket high_basket_0: Retrieved 197 documents with 223 chunks
2026-01-27 13:28:28,860 - INFO - Basket high_basket_4: Retrieved 79 documents with 113 chunks
2026-01-27 13:28:29,389 